# Contextualize HaeconGEM1 with transcriptomic data

Run from repo root. Requires: `pip install git+https://github.com/fma91/riptide.git`

In [ ]:
from pathlib import Path
import cobra
import pandas as pd
import riptide

# Paths (relative to repo root)
BASE_MODEL = Path('model/haeconGEM1.xml')
TPM_FOLDER = Path('input_data/TPM')
OUTPUT_MODELS = Path('model/contextualized_models')
OUTPUT_MODELS.mkdir(parents=True, exist_ok=True)

# Stages to contextualize
STAGES = ['egg', 'L1', 'L2', 'L3', 'L4f', 'L4m', 'Am', 'Af', 'L4mf_union']

print(f'RIPTiDe: {riptide.__file__}')
print(f'Base model: {BASE_MODEL.resolve()}')

In [ ]:
# Load reference model once
base_model = cobra.io.read_sbml_model(str(BASE_MODEL))
print(f'{base_model.id}: {len(base_model.genes)} genes, {len(base_model.reactions)} reactions')

In [ ]:
# Contextualize each stage
for stage in STAGES:
    output_path = OUTPUT_MODELS / f'{stage}_c.xml'
    
    # Skip if already exists
    if output_path.exists():
        print(f'{stage}: already exists, skipping')
        continue
    
    # Load transcriptome
    tpm_file = TPM_FOLDER / f'TPM_{stage}_GEM.csv'
    transcriptome = riptide.read_transcription_file(str(tpm_file), sep='\t')
    
    # Contextualize
    result = riptide.contextualize(model=base_model.copy(), transcriptome=transcriptome)
    result.model.id = f'{stage}_c'
    result.model.name = f'{stage}_c'
    
    # Save
    cobra.io.write_sbml_model(result.model, str(output_path))
    print(f'{stage}: {len(result.model.reactions)} reactions -> {output_path.name}')

In [ ]:
# Build inventory tables
gene_rows = []
metabolite_rows = []
reaction_rows = []

for stage in STAGES:
    model_path = OUTPUT_MODELS / f'{stage}_c.xml'
    if not model_path.exists():
        continue
    
    model = cobra.io.read_sbml_model(str(model_path))
    
    for gene in model.genes:
        gene_rows.append({'stage': stage, 'gene_id': gene.id, 'gene_name': gene.name, 'n_reactions': len(gene.reactions)})
    
    for met in model.metabolites:
        metabolite_rows.append({'stage': stage, 'metabolite_id': met.id, 'metabolite_name': met.name, 
                                'formula': met.formula or '', 'compartment': met.compartment, 'n_reactions': len(met.reactions)})
    
    for rxn in model.reactions:
        reaction_rows.append({'stage': stage, 'reaction_id': rxn.id, 'reaction_name': rxn.name, 
                              'subsystem': rxn.subsystem or 'Unassigned', 'lower_bound': rxn.lower_bound, 
                              'upper_bound': rxn.upper_bound, 'gene_reaction_rule': rxn.gene_reaction_rule,
                              'n_genes': len(rxn.genes), 'n_metabolites': len(rxn.metabolites)})

# Save tables
pd.DataFrame(gene_rows).to_csv('contextualized_model_genes_all_stages.csv', index=False)
pd.DataFrame(metabolite_rows).to_csv('contextualized_model_metabolites_all_stages.csv', index=False)
pd.DataFrame(reaction_rows).to_csv('contextualized_model_reactions_all_stages.csv', index=False)

print(f'Saved inventory tables ({len(gene_rows)} genes, {len(metabolite_rows)} metabolites, {len(reaction_rows)} reactions across all stages)')